In [1]:
pip install lightgbm optuna

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.0/1.5 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------------ --------------- 1.3/2.1 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 7.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Todo importado correctamente.')

Todo importado correctamente.


In [3]:
train = pd.read_csv('train_clean.csv')
test = pd.read_csv('test_nolabel.csv')

X_test_df = test.copy()

# Aplicar las mismas transformaciones que en el train
MIN_COUNT = 5
vc = train['speaker'].value_counts()
speakers_frecuentes = vc[vc >= MIN_COUNT].index
X_test_df['speaker_grouped'] = test['speaker'].where(test['speaker'].isin(speakers_frecuentes), other='other')

vc_party = train['party_affiliation'].value_counts()
minoritarias = vc_party[vc_party < 50].index
X_test_df['party_grouped'] = test['party_affiliation'].where(~test['party_affiliation'].isin(minoritarias), other='other')

X_test_df['subject_primary'] = test['subject'].str.split(',').str[0].str.strip()

y_train = train['label']

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print()
print('Distribución label:')
print(y_train.value_counts())

Train shape: (8947, 10)
Test shape: (3836, 7)

Distribución label:
label
1    5794
0    3153
Name: count, dtype: int64


In [4]:
# TF-IDF a nivel de palabra (1-2 gramas)
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=50000,
    sublinear_tf=True
)

# TF-IDF a nivel de caracter (3-5 gramas)
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=3,
    max_features=50000,
    sublinear_tf=True
)

# Fit en train, transform en train y test
X_train_word = tfidf_word.fit_transform(train['statement'])
X_test_word = tfidf_word.transform(X_test_df['statement'])

X_train_char = tfidf_char.fit_transform(train['statement'])
X_test_char = tfidf_char.transform(X_test_df['statement'])

print('TF-IDF word shape:', X_train_word.shape)
print('TF-IDF char shape:', X_train_char.shape)

TF-IDF word shape: (8947, 13196)
TF-IDF char shape: (8947, 34208)


In [5]:
# Combinar word + char antes de reducir
X_train_text = hstack([X_train_word, X_train_char])
X_test_text = hstack([X_test_word, X_test_char])

print('Shape combinado antes de SVD:', X_train_text.shape)

# Reducir a 300 dimensiones
svd = TruncatedSVD(n_components=300, random_state=42)
X_train_svd = svd.fit_transform(X_train_text)
X_test_svd = svd.transform(X_test_text)

varianza_explicada = svd.explained_variance_ratio_.sum()
print(f'Shape tras SVD: {X_train_svd.shape}')
print(f'Varianza explicada: {varianza_explicada:.3f} ({varianza_explicada*100:.1f}%)')

Shape combinado antes de SVD: (8947, 47404)
Shape tras SVD: (8947, 300)
Varianza explicada: 0.326 (32.6%)


In [6]:
for n in [100, 200, 300, 400, 500]:
    svd_test = TruncatedSVD(n_components=n, random_state=42)
    svd_test.fit(X_train_text)
    var = svd_test.explained_variance_ratio_.sum()
    print(f'n_components={n}: varianza explicada={var:.3f} ({var*100:.1f}%)')

n_components=100: varianza explicada=0.172 (17.2%)
n_components=200: varianza explicada=0.261 (26.1%)
n_components=300: varianza explicada=0.326 (32.6%)
n_components=400: varianza explicada=0.378 (37.8%)
n_components=500: varianza explicada=0.421 (42.1%)


In [7]:
# SVD con 500 componentes
svd = TruncatedSVD(n_components=500, random_state=42)
X_train_svd = svd.fit_transform(X_train_text)
X_test_svd = svd.transform(X_test_text)

print(f'Shape SVD: {X_train_svd.shape}')
print(f'Varianza explicada: {svd.explained_variance_ratio_.sum():.3f}')
print()

# Añadir features categóricas
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
cat_cols = ['speaker_grouped', 'party_grouped', 'subject_primary']
X_train_cat = ohe.fit_transform(train[cat_cols])
X_test_cat = ohe.transform(X_test_df[cat_cols])

# Combinar SVD + categóricas
X_train_final = np.hstack([X_train_svd, X_train_cat])
X_test_final = np.hstack([X_test_svd, X_test_cat])

print(f'Shape final train: {X_train_final.shape}')
print(f'Shape final test: {X_test_final.shape}')

Shape SVD: (8947, 500)
Varianza explicada: 0.421

Shape final train: (8947, 949)
Shape final test: (3836, 949)


In [8]:
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X_train_final, y_train):
        X_tr, X_val = X_train_final[train_idx], X_train_final[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=500)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])
        y_pred = model.predict(X_val)
        scores.append(f1_score(y_val, y_pred, average='macro'))

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nMejor F1-Macro: {study.best_value:.4f}')
print('Mejores parámetros:')
for param, valor in study.best_params.items():
    print(f'  {param}: {valor}')

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor F1-Macro: 0.5316
Mejores parámetros:
  num_leaves: 148
  max_depth: 7
  learning_rate: 0.07949075217922058
  feature_fraction: 0.9518201386930145
  bagging_fraction: 0.5716286098495298
  bagging_freq: 3
  min_data_in_leaf: 5
  lambda_l1: 1.0919248621529134e-06
  lambda_l2: 9.597985231963285e-06


In [9]:
# Combinar TF-IDF word + char + categóricas sin reducir dimensionalidad
from scipy.sparse import csr_matrix

X_train_cat_sparse = csr_matrix(X_train_cat)
X_test_cat_sparse = csr_matrix(X_test_cat)

X_train_nosvd = hstack([X_train_text, X_train_cat_sparse])
X_test_nosvd = hstack([X_test_text, X_test_cat_sparse])

print(f'Shape train sin SVD: {X_train_nosvd.shape}')
print(f'Shape test sin SVD: {X_test_nosvd.shape}')

# Evaluación rápida con parámetros por defecto
modelo_base = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=100,
    random_state=42,
    verbosity=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for train_idx, val_idx in cv.split(X_train_nosvd, y_train):
    X_tr, X_val = X_train_nosvd[train_idx], X_train_nosvd[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    modelo_base.fit(X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(-1)])
    y_pred = modelo_base.predict(X_val)
    scores.append(f1_score(y_val, y_pred, average='macro'))

print(f'\nF1-Macro sin SVD: {np.mean(scores):.4f}')
print(f'F1-Macro con SVD: 0.5316')

Shape train sin SVD: (8947, 47853)
Shape test sin SVD: (3836, 47853)

F1-Macro sin SVD: 0.5393
F1-Macro con SVD: 0.5316


In [10]:
def objective_v2(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'is_unbalance': True,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X_train_nosvd, y_train):
        X_tr, X_val = X_train_nosvd[train_idx], X_train_nosvd[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=500)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])
        y_pred = model.predict(X_val)
        scores.append(f1_score(y_val, y_pred, average='macro'))

    return np.mean(scores)

study_v2 = optuna.create_study(direction='maximize')
study_v2.optimize(objective_v2, n_trials=50, show_progress_bar=True)

print(f'\nMejor F1-Macro: {study_v2.best_value:.4f}')
print('Mejores parámetros:')
for param, valor in study_v2.best_params.items():
    print(f'  {param}: {valor}')

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor F1-Macro: 0.5135
Mejores parámetros:
  num_leaves: 77
  max_depth: 8
  learning_rate: 0.09647007771737161
  feature_fraction: 0.8440532646823248
  bagging_fraction: 0.8393583498309456
  bagging_freq: 2
  min_data_in_leaf: 42
  lambda_l1: 1.6838507996834614e-05
  lambda_l2: 3.9163201083316364e-08


In [11]:
def text_features(df_col):
    features = pd.DataFrame()
    features['length'] = df_col.str.len()
    features['word_count'] = df_col.str.split().str.len()
    features['avg_word_length'] = df_col.apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)
    features['exclamation'] = df_col.str.count('!')
    features['question'] = df_col.str.count('\?')
    features['uppercase_ratio'] = df_col.apply(lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0)
    features['digit_count'] = df_col.str.count('\d')
    features['comma_count'] = df_col.str.count(',')
    return features.values

X_train_manual = text_features(train['statement'])
X_test_manual = text_features(X_test_df['statement'])

print('Features manuales shape:', X_train_manual.shape)
print()
cols = ['length','word_count','avg_word_length','exclamation','question','uppercase_ratio','digit_count','comma_count']
print(pd.DataFrame(X_train_manual, columns=cols).head(3).to_string())

Features manuales shape: (8947, 8)

   length  word_count  avg_word_length  exclamation  question  uppercase_ratio  digit_count  comma_count
0   106.0        21.0         4.095238          0.0       0.0         0.037736          0.0          0.0
1   164.0        30.0         4.500000          0.0       0.0         0.018293          0.0          1.0
2    57.0         8.0         6.125000          0.0       0.0         0.017544          0.0          0.0


In [13]:
from scipy.sparse import csr_matrix, hstack

# TF-IDF word sin SVD
tfidf_word_v2 = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=30000,
    sublinear_tf=True
)

X_train_tfidf = tfidf_word_v2.fit_transform(train['statement'])
X_test_tfidf = tfidf_word_v2.transform(X_test_df['statement'])

# Convertir features manuales y categóricas a sparse
X_train_manual_sparse = csr_matrix(X_train_manual)
X_test_manual_sparse = csr_matrix(X_test_manual)

X_train_cat_sparse = csr_matrix(X_train_cat)
X_test_cat_sparse = csr_matrix(X_test_cat)

# Combinar todo
X_train_v2 = hstack([X_train_tfidf, X_train_manual_sparse, X_train_cat_sparse])
X_test_v2 = hstack([X_test_tfidf, X_test_manual_sparse, X_test_cat_sparse])

print('Shape train final:', X_train_v2.shape)
print('Shape test final:', X_test_v2.shape)
print()
print('Desglose de features:')
print(f'  TF-IDF word:      {X_train_tfidf.shape[1]}')
print(f'  Features manuales: {X_train_manual_sparse.shape[1]}')
print(f'  Features categóricas: {X_train_cat_sparse.shape[1]}')
print(f'  Total:            {X_train_v2.shape[1]}')

Shape train final: (8947, 13653)
Shape test final: (3836, 13653)

Desglose de features:
  TF-IDF word:      13196
  Features manuales: 8
  Features categóricas: 449
  Total:            13653


In [14]:
def objective_v3(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'is_unbalance': True,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X_train_v2, y_train):
        X_tr, X_val = X_train_v2[train_idx], X_train_v2[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=500)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])
        y_pred = model.predict(X_val)
        scores.append(f1_score(y_val, y_pred, average='macro'))

    return np.mean(scores)

study_v3 = optuna.create_study(direction='maximize')
study_v3.optimize(objective_v3, n_trials=50, show_progress_bar=True)

print(f'\nMejor F1-Macro: {study_v3.best_value:.4f}')
print('Mejores parámetros:')
for param, valor in study_v3.best_params.items():
    print(f'  {param}: {valor}')
    

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor F1-Macro: 0.5595
Mejores parámetros:
  num_leaves: 147
  max_depth: 12
  learning_rate: 0.032969920889647064
  feature_fraction: 0.6774344247397508
  bagging_fraction: 0.737662594112799
  bagging_freq: 1
  min_data_in_leaf: 8
  lambda_l1: 2.5224439691995507e-08
  lambda_l2: 5.714929905275524e-05


In [15]:
# Entrenar modelo con mejores parámetros
best_params = study_v3.best_params
best_params.update({
    'objective': 'binary',
    'verbosity': -1,
    'is_unbalance': True,
    'random_state': 42
})

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba_lgb = np.zeros(len(y_train))

for train_idx, val_idx in cv.split(X_train_v2, y_train):
    X_tr, X_val = X_train_v2[train_idx], X_train_v2[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**best_params, n_estimators=500)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
    y_proba_lgb[val_idx] = model.predict_proba(X_val)[:, 1]

print('Threshold tuning:')
print()
for threshold in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_pred_thresh = (y_proba_lgb >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_thresh, average='macro')
    f1_fake = f1_score(y_train, y_pred_thresh, average=None)[0]
    f1_real = f1_score(y_train, y_pred_thresh, average=None)[1]
    print(f'Threshold {threshold:.2f}: F1-Macro={f1:.4f} | F1-fake={f1_fake:.4f} | F1-real={f1_real:.4f}')

Threshold tuning:

Threshold 0.35: F1-Macro=0.4346 | F1-fake=0.0820 | F1-real=0.7872
Threshold 0.40: F1-Macro=0.4630 | F1-fake=0.1405 | F1-real=0.7855
Threshold 0.45: F1-Macro=0.5088 | F1-fake=0.2383 | F1-real=0.7793
Threshold 0.50: F1-Macro=0.5710 | F1-fake=0.3851 | F1-real=0.7570
Threshold 0.55: F1-Macro=0.6008 | F1-fake=0.4962 | F1-real=0.7055
Threshold 0.60: F1-Macro=0.5652 | F1-fake=0.5543 | F1-real=0.5760


In [16]:
# Añadir char TF-IDF directamente sin SVD
tfidf_char_v2 = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=5,
    max_features=30000,
    sublinear_tf=True
)

X_train_char_v2 = tfidf_char_v2.fit_transform(train['statement'])
X_test_char_v2 = tfidf_char_v2.transform(X_test_df['statement'])

# Combinar con las features anteriores
X_train_v3 = hstack([X_train_tfidf, X_train_char_v2, X_train_manual_sparse, X_train_cat_sparse])
X_test_v3 = hstack([X_test_tfidf, X_test_char_v2, X_test_manual_sparse, X_test_cat_sparse])

print('Shape train con char TF-IDF:', X_train_v3.shape)
print('Shape test con char TF-IDF:', X_test_v3.shape)
print()
print('Desglose:')
print(f'  TF-IDF word:      {X_train_tfidf.shape[1]}')
print(f'  TF-IDF char:      {X_train_char_v2.shape[1]}')
print(f'  Features manuales: {X_train_manual_sparse.shape[1]}')
print(f'  Features categóricas: {X_train_cat_sparse.shape[1]}')
print(f'  Total:            {X_train_v3.shape[1]}')

Shape train con char TF-IDF: (8947, 39465)
Shape test con char TF-IDF: (3836, 39465)

Desglose:
  TF-IDF word:      13196
  TF-IDF char:      25812
  Features manuales: 8
  Features categóricas: 449
  Total:            39465


In [17]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba_v3 = np.zeros(len(y_train))

for train_idx, val_idx in cv.split(X_train_v3, y_train):
    X_tr, X_val = X_train_v3[train_idx], X_train_v3[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**best_params, n_estimators=500)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
    y_proba_v3[val_idx] = model.predict_proba(X_val)[:, 1]

print('Threshold tuning con char TF-IDF:')
print()
for threshold in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    y_pred_thresh = (y_proba_v3 >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_thresh, average='macro')
    f1_fake = f1_score(y_train, y_pred_thresh, average=None)[0]
    f1_real = f1_score(y_train, y_pred_thresh, average=None)[1]
    print(f'Threshold {threshold:.2f}: F1-Macro={f1:.4f} | F1-fake={f1_fake:.4f} | F1-real={f1_real:.4f}')

Threshold tuning con char TF-IDF:

Threshold 0.40: F1-Macro=0.4425 | F1-fake=0.0974 | F1-real=0.7876
Threshold 0.45: F1-Macro=0.5015 | F1-fake=0.2212 | F1-real=0.7819
Threshold 0.50: F1-Macro=0.5695 | F1-fake=0.3821 | F1-real=0.7570
Threshold 0.55: F1-Macro=0.5951 | F1-fake=0.4911 | F1-real=0.6990
Threshold 0.60: F1-Macro=0.5580 | F1-fake=0.5399 | F1-real=0.5761
Threshold 0.65: F1-Macro=0.4578 | F1-fake=0.5451 | F1-real=0.3706


In [18]:
def objective_v4(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'is_unbalance': True,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X_train_v3, y_train):
        X_tr, X_val = X_train_v3[train_idx], X_train_v3[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=500)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])
        y_pred = model.predict(X_val)
        scores.append(f1_score(y_val, y_pred, average='macro'))

    return np.mean(scores)

study_v4 = optuna.create_study(direction='maximize')
study_v4.optimize(objective_v4, n_trials=50, show_progress_bar=True)

print(f'\nMejor F1-Macro: {study_v4.best_value:.4f}')
print('Mejores parámetros:')
for param, valor in study_v4.best_params.items():
    print(f'  {param}: {valor}')

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor F1-Macro: 0.5650
Mejores parámetros:
  num_leaves: 118
  max_depth: 11
  learning_rate: 0.021410438910766928
  feature_fraction: 0.756977716959805
  bagging_fraction: 0.6739852403906742
  bagging_freq: 4
  min_data_in_leaf: 22
  lambda_l1: 4.296302490813517e-05
  lambda_l2: 1.0798306125728225


In [19]:
best_params_v4 = study_v4.best_params
best_params_v4.update({
    'objective': 'binary',
    'verbosity': -1,
    'is_unbalance': True,
    'random_state': 42
})

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba_v4 = np.zeros(len(y_train))

for train_idx, val_idx in cv.split(X_train_v3, y_train):
    X_tr, X_val = X_train_v3[train_idx], X_train_v3[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**best_params_v4, n_estimators=500)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
    y_proba_v4[val_idx] = model.predict_proba(X_val)[:, 1]

print('Threshold tuning parámetros optimizados + char TF-IDF:')
print()
for threshold in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    y_pred_thresh = (y_proba_v4 >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_thresh, average='macro')
    f1_fake = f1_score(y_train, y_pred_thresh, average=None)[0]
    f1_real = f1_score(y_train, y_pred_thresh, average=None)[1]
    print(f'Threshold {threshold:.2f}: F1-Macro={f1:.4f} | F1-fake={f1_fake:.4f} | F1-real={f1_real:.4f}')

Threshold tuning parámetros optimizados + char TF-IDF:

Threshold 0.40: F1-Macro=0.4533 | F1-fake=0.1220 | F1-real=0.7846
Threshold 0.45: F1-Macro=0.5070 | F1-fake=0.2332 | F1-real=0.7807
Threshold 0.50: F1-Macro=0.5731 | F1-fake=0.3862 | F1-real=0.7600
Threshold 0.55: F1-Macro=0.5987 | F1-fake=0.4983 | F1-real=0.6992
Threshold 0.60: F1-Macro=0.5614 | F1-fake=0.5488 | F1-real=0.5739
Threshold 0.65: F1-Macro=0.4530 | F1-fake=0.5491 | F1-real=0.3569


In [ ]:
# Obtener probabilidades del Naive Bayes sobre el train
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack as sp_hstack

# Reconstruir el mejor modelo NB
vect_nb = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=5)
X_train_nb = sp_hstack([vect_nb.fit_transform(train['statement']), csr_matrix(X_train_cat)])
X_test_nb = sp_hstack([vect_nb.transform(X_test_df['statement']), csr_matrix(X_test_cat)])

nb_model = ComplementNB(alpha=2.0, norm=False)

# Probabilidades CV del NB
y_proba_nb_cv = np.zeros(len(y_train))
for train_idx, val_idx in cv.split(X_train_nb, y_train):
    nb_model.fit(X_train_nb[train_idx], y_train.iloc[train_idx])
    y_proba_nb_cv[val_idx] = nb_model.predict_proba(X_train_nb[val_idx])[:, 1]

print('Probar distintos pesos ensemble NB + LightGBM:')
print()
for w_nb in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    w_lgb = 1 - w_nb
    y_proba_ensemble = w_nb * y_proba_nb_cv + w_lgb * y_proba_lgb

    best_f1 = 0
    best_thresh = 0.5
    for threshold in [0.40, 0.45, 0.50, 0.55, 0.60]:
        y_pred = (y_proba_ensemble >= threshold).astype(int)
        f1 = f1_score(y_train, y_pred, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = threshold

    print(f'NB={w_nb:.1f} LGB={w_lgb:.1f}: F1-Macro={best_f1:.4f} (threshold={best_thresh})')

Probar distintos pesos ensemble NB + LightGBM:

NB=0.2 LGB=0.8: F1-Macro=0.6127 (threshold=0.55)
NB=0.3 LGB=0.7: F1-Macro=0.6123 (threshold=0.5)
NB=0.4 LGB=0.6: F1-Macro=0.6108 (threshold=0.5)
NB=0.5 LGB=0.5: F1-Macro=0.6092 (threshold=0.45)
NB=0.6 LGB=0.4: F1-Macro=0.6089 (threshold=0.45)
NB=0.7 LGB=0.3: F1-Macro=0.6073 (threshold=0.45)


In [21]:
# Entrenar NB final con todos los datos
nb_model.fit(X_train_nb, y_train)
y_proba_nb_test = nb_model.predict_proba(X_test_nb)[:, 1]

# Entrenar LightGBM final con todos los datos
modelo_lgb_final = lgb.LGBMClassifier(**best_params, n_estimators=500)
modelo_lgb_final.fit(X_train_v2, y_train, callbacks=[lgb.log_evaluation(-1)])
y_proba_lgb_test = modelo_lgb_final.predict_proba(X_test_v2)[:, 1]

# Ensemble con pesos óptimos
y_proba_ensemble_test = 0.2 * y_proba_nb_test + 0.8 * y_proba_lgb_test
y_pred_ensemble = (y_proba_ensemble_test >= 0.55).astype(int)

# Submission
submission = pd.DataFrame({
    'id': test['id'],
    'label': y_pred_ensemble
})

submission.to_csv('submission_lightgbm.csv', index=False)
print('Submission guardada como submission_lightgbm.csv')
print()
print('Shape:', submission.shape)
print()
print('Distribución de predicciones:')
print(submission['label'].value_counts())
print()
print('Resumen final:')
print(f'  Modelo:          Ensemble LightGBM + Naive Bayes')
print(f'  Pesos:           LGB=0.8, NB=0.2')
print(f'  Features LGB:    TF-IDF word (1,2) + manuales + categóricas (13.653)')
print(f'  Features NB:     CountVectorizer bigrams + categóricas')
print(f'  Optimización:    Optuna 50 trials')
print(f'  Threshold:       0.55')
print(f'  F1-Macro CV:     0.6127')

Submission guardada como submission_lightgbm.csv

Shape: (3836, 2)

Distribución de predicciones:
label
1    2049
0    1787
Name: count, dtype: int64

Resumen final:
  Modelo:          Ensemble LightGBM + Naive Bayes
  Pesos:           LGB=0.8, NB=0.2
  Features LGB:    TF-IDF word (1,2) + manuales + categóricas (13.653)
  Features NB:     CountVectorizer bigrams + categóricas
  Optimización:    Optuna 50 trials
  Threshold:       0.55
  F1-Macro CV:     0.6127


In [22]:
study_v5 = optuna.create_study(direction='maximize')
study_v5.optimize(objective_v3, n_trials=100, show_progress_bar=True)

print(f'\nMejor F1-Macro (100 trials): {study_v5.best_value:.4f}')
print(f'Mejor F1-Macro (50 trials):  0.5595')
print(f'Diferencia:                  {study_v5.best_value - 0.5595:+.4f}')
print()
print('Mejores parámetros:')
for param, valor in study_v5.best_params.items():
    print(f'  {param}: {valor}')

  0%|          | 0/100 [00:00<?, ?it/s]


Mejor F1-Macro (100 trials): 0.5474
Mejor F1-Macro (50 trials):  0.5595
Diferencia:                  -0.0121

Mejores parámetros:
  num_leaves: 95
  max_depth: 12
  learning_rate: 0.05140313608504135
  feature_fraction: 0.9977984761151821
  bagging_fraction: 0.9164768948706038
  bagging_freq: 7
  min_data_in_leaf: 20
  lambda_l1: 0.002260312496924859
  lambda_l2: 9.7775699353172e-06


In [23]:
def objective_dart(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'dart',
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'drop_rate': trial.suggest_float('drop_rate', 0.1, 0.5),
        'is_unbalance': True,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X_train_v2, y_train):
        X_tr, X_val = X_train_v2[train_idx], X_train_v2[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=300)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.log_evaluation(-1)])
        y_pred = model.predict(X_val)
        scores.append(f1_score(y_val, y_pred, average='macro'))

    return np.mean(scores)

study_dart = optuna.create_study(direction='maximize')
study_dart.optimize(objective_dart, n_trials=30, show_progress_bar=True)

print(f'\nMejor F1-Macro (dart): {study_dart.best_value:.4f}')
print(f'Mejor F1-Macro (gbdt): 0.5595')
print(f'Diferencia:            {study_dart.best_value - 0.5595:+.4f}')

  0%|          | 0/30 [00:00<?, ?it/s]


Mejor F1-Macro (dart): 0.6111
Mejor F1-Macro (gbdt): 0.5595
Diferencia:            +0.0516


In [24]:
best_params_dart = study_dart.best_params
best_params_dart.update({
    'objective': 'binary',
    'verbosity': -1,
    'boosting_type': 'dart',
    'is_unbalance': True,
    'random_state': 42
})

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba_dart = np.zeros(len(y_train))

for train_idx, val_idx in cv.split(X_train_v2, y_train):
    X_tr, X_val = X_train_v2[train_idx], X_train_v2[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = lgb.LGBMClassifier(**best_params_dart, n_estimators=300)
    model.fit(X_tr, y_tr, callbacks=[lgb.log_evaluation(-1)])
    y_proba_dart[val_idx] = model.predict_proba(X_val)[:, 1]

print('Threshold tuning DART:')
print()
for threshold in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    y_pred_thresh = (y_proba_dart >= threshold).astype(int)
    f1 = f1_score(y_train, y_pred_thresh, average='macro')
    f1_fake = f1_score(y_train, y_pred_thresh, average=None)[0]
    f1_real = f1_score(y_train, y_pred_thresh, average=None)[1]
    print(f'Threshold {threshold:.2f}: F1-Macro={f1:.4f} | F1-fake={f1_fake:.4f} | F1-real={f1_real:.4f}')
    

Threshold tuning DART:

Threshold 0.35: F1-Macro=0.4323 | F1-fake=0.0792 | F1-real=0.7854
Threshold 0.40: F1-Macro=0.5105 | F1-fake=0.2437 | F1-real=0.7773
Threshold 0.45: F1-Macro=0.5886 | F1-fake=0.4245 | F1-real=0.7526
Threshold 0.50: F1-Macro=0.6111 | F1-fake=0.5290 | F1-real=0.6932
Threshold 0.55: F1-Macro=0.5526 | F1-fake=0.5541 | F1-real=0.5511
Threshold 0.60: F1-Macro=0.4421 | F1-fake=0.5470 | F1-real=0.3372
Threshold 0.65: F1-Macro=0.3557 | F1-fake=0.5371 | F1-real=0.1742


In [25]:
print('Probar distintos pesos ensemble NB + LightGBM DART:')
print()
for w_nb in [0.1, 0.2, 0.3, 0.4, 0.5]:
    w_lgb = 1 - w_nb
    y_proba_ensemble_dart = w_nb * y_proba_nb_cv + w_lgb * y_proba_dart

    best_f1 = 0
    best_thresh = 0.5
    for threshold in [0.40, 0.45, 0.50, 0.55, 0.60]:
        y_pred = (y_proba_ensemble_dart >= threshold).astype(int)
        f1 = f1_score(y_train, y_pred, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = threshold

    print(f'NB={w_nb:.1f} LGB={w_lgb:.1f}: F1-Macro={best_f1:.4f} (threshold={best_thresh})')

Probar distintos pesos ensemble NB + LightGBM DART:

NB=0.1 LGB=0.9: F1-Macro=0.6150 (threshold=0.5)
NB=0.2 LGB=0.8: F1-Macro=0.6136 (threshold=0.5)
NB=0.3 LGB=0.7: F1-Macro=0.6112 (threshold=0.45)
NB=0.4 LGB=0.6: F1-Macro=0.6096 (threshold=0.45)
NB=0.5 LGB=0.5: F1-Macro=0.6099 (threshold=0.45)


In [26]:
print('Afinar pesos ensemble NB + DART:')
print()
for w_nb in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
    w_lgb = 1 - w_nb
    y_proba_ensemble_dart = w_nb * y_proba_nb_cv + w_lgb * y_proba_dart

    best_f1 = 0
    best_thresh = 0.5
    for threshold in [0.42, 0.45, 0.47, 0.50, 0.52, 0.55]:
        y_pred = (y_proba_ensemble_dart >= threshold).astype(int)
        f1 = f1_score(y_train, y_pred, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = threshold

    print(f'NB={w_nb:.2f} LGB={w_lgb:.2f}: F1-Macro={best_f1:.4f} (threshold={best_thresh})')

Afinar pesos ensemble NB + DART:

NB=0.05 LGB=0.95: F1-Macro=0.6144 (threshold=0.5)
NB=0.08 LGB=0.92: F1-Macro=0.6172 (threshold=0.5)
NB=0.10 LGB=0.90: F1-Macro=0.6156 (threshold=0.47)
NB=0.12 LGB=0.88: F1-Macro=0.6160 (threshold=0.47)
NB=0.15 LGB=0.85: F1-Macro=0.6153 (threshold=0.47)
NB=0.20 LGB=0.80: F1-Macro=0.6146 (threshold=0.47)


In [27]:
# Entrenar DART final con todos los datos
modelo_dart_final = lgb.LGBMClassifier(**best_params_dart, n_estimators=300)
modelo_dart_final.fit(X_train_v2, y_train, callbacks=[lgb.log_evaluation(-1)])
y_proba_dart_test = modelo_dart_final.predict_proba(X_test_v2)[:, 1]

# Ensemble con pesos óptimos
y_proba_ensemble_final = 0.08 * y_proba_nb_test + 0.92 * y_proba_dart_test
y_pred_final = (y_proba_ensemble_final >= 0.50).astype(int)

# Submission
submission = pd.DataFrame({
    'id': test['id'],
    'label': y_pred_final
})

submission.to_csv('submission_lightgbm.csv', index=False)
print('Submission actualizada como submission_lightgbm.csv')
print()
print('Distribución de predicciones:')
print(submission['label'].value_counts())
print()
print('Resumen final:')
print(f'  Modelo:          Ensemble LightGBM DART + Naive Bayes')
print(f'  Pesos:           LGB=0.92, NB=0.08')
print(f'  Features LGB:    TF-IDF word (1,2) + manuales + categóricas (13.653)')
print(f'  Features NB:     CountVectorizer bigrams + categóricas')
print(f'  boosting_type:   dart')
print(f'  Optimización:    Optuna 30 trials')
print(f'  Threshold:       0.50')
print(f'  F1-Macro CV:     0.6172')

Submission actualizada como submission_lightgbm.csv

Distribución de predicciones:
label
1    2198
0    1638
Name: count, dtype: int64

Resumen final:
  Modelo:          Ensemble LightGBM DART + Naive Bayes
  Pesos:           LGB=0.92, NB=0.08
  Features LGB:    TF-IDF word (1,2) + manuales + categóricas (13.653)
  Features NB:     CountVectorizer bigrams + categóricas
  boosting_type:   dart
  Optimización:    Optuna 30 trials
  Threshold:       0.50
  F1-Macro CV:     0.6172


In [28]:
# Submission solo DART sin ensemble
y_pred_dart_solo = (y_proba_dart_test >= 0.50).astype(int)

submission_dart = pd.DataFrame({
    'id': test['id'],
    'label': y_pred_dart_solo
})

submission_dart.to_csv('submission_lightgbm_dart_solo.csv', index=False)
print('Submission DART solo guardada.')
print()
print('Distribución:')
print(submission_dart['label'].value_counts())

Submission DART solo guardada.

Distribución:
label
1    2128
0    1708
Name: count, dtype: int64


In [29]:
# Reconstruir NB mejorado (alpha=3.5, min_df=7)
vect_nb_best = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=7)
X_train_nb_best = sp_hstack([vect_nb_best.fit_transform(train['statement']), csr_matrix(X_train_cat)])
X_test_nb_best = sp_hstack([vect_nb_best.transform(X_test_df['statement']), csr_matrix(X_test_cat)])

nb_model_best = ComplementNB(alpha=3.5, norm=False)

# Probabilidades CV del NB mejorado
y_proba_nb_best_cv = np.zeros(len(y_train))
for train_idx, val_idx in cv.split(X_train_nb_best, y_train):
    nb_model_best.fit(X_train_nb_best[train_idx], y_train.iloc[train_idx])
    y_proba_nb_best_cv[val_idx] = nb_model_best.predict_proba(X_train_nb_best[val_idx])[:, 1]

print('Ensemble NB mejorado + DART:')
print()
for w_nb in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
    w_lgb = 1 - w_nb
    y_proba_ens = w_nb * y_proba_nb_best_cv + w_lgb * y_proba_dart

    best_f1 = 0
    best_thresh = 0.5
    for threshold in [0.42, 0.45, 0.47, 0.50, 0.52, 0.55]:
        y_pred = (y_proba_ens >= threshold).astype(int)
        f1 = f1_score(y_train, y_pred, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = threshold

    print(f'NB={w_nb:.2f} LGB={w_lgb:.2f}: F1-Macro={best_f1:.4f} (threshold={best_thresh})')

Ensemble NB mejorado + DART:

NB=0.05 LGB=0.95: F1-Macro=0.6159 (threshold=0.5)
NB=0.08 LGB=0.92: F1-Macro=0.6166 (threshold=0.5)
NB=0.10 LGB=0.90: F1-Macro=0.6171 (threshold=0.5)
NB=0.12 LGB=0.88: F1-Macro=0.6178 (threshold=0.47)
NB=0.15 LGB=0.85: F1-Macro=0.6175 (threshold=0.5)
NB=0.20 LGB=0.80: F1-Macro=0.6183 (threshold=0.5)


In [31]:
# Entrenar NB mejorado final
nb_model_best.fit(X_train_nb_best, y_train)
y_proba_nb_best_test = nb_model_best.predict_proba(X_test_nb_best)[:, 1]

# Ensemble con pesos óptimos
y_proba_ensemble_best = 0.20 * y_proba_nb_best_test + 0.80 * y_proba_dart_test
y_pred_ensemble_best = (y_proba_ensemble_best >= 0.50).astype(int)

submission = pd.DataFrame({
    'id': test['id'],
    'label': y_pred_ensemble_best
})

submission.to_csv('submission_lightgbm.csv', index=False)
print('Submission guardada.')
print()
print('Distribución:')
print(submission['label'].value_counts())
print()
print('Resumen final:')
print(f'  Modelo:        Ensemble LightGBM DART + NB mejorado')
print(f'  Pesos:         LGB=0.80, NB=0.20')
print(f'  boosting_type: dart')
print(f'  Threshold:     0.50')
print(f'  F1-Macro CV:   0.6183')

Submission guardada.

Distribución:
label
1    2251
0    1585
Name: count, dtype: int64

Resumen final:
  Modelo:        Ensemble LightGBM DART + NB mejorado
  Pesos:         LGB=0.80, NB=0.20
  boosting_type: dart
  Threshold:     0.50
  F1-Macro CV:   0.6183
